# PELIC — Load & Filter by L1
This notebook clones the PELIC dataset, inspects L1 coverage, and exports per-L1 text files ready for stylometric feature extraction.

## 1. Clone the repo (run once)

In [26]:
import urllib.request, os

os.makedirs("PELIC-dataset", exist_ok=True)

url = "https://media.githubusercontent.com/media/ELI-Data-Mining-Group/PELIC-dataset/master/PELIC_compiled.csv"
dest = "PELIC-dataset/PELIC_compiled.csv"

if not os.path.exists(dest) or os.path.getsize(dest) < 1_000_000:
    print("Downloading PELIC_compiled.csv (~180MB), this may take a minute...")
    urllib.request.urlretrieve(url, dest)
    print("Done.")
else:
    print("Already downloaded.")

Done.


## 2. Load the compiled CSV

In [28]:
import pandas as pd

df = pd.read_csv("PELIC-dataset/PELIC_compiled.csv")
print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

Total rows: 46,204
Columns: ['answer_id', 'anon_id', 'L1', 'gender', 'semester', 'placement_test', 'course_id', 'level_id', 'class_id', 'question_id', 'version', 'text_len', 'text', 'tokens', 'tok_lem_POS']


,answer_id,anon_id,L1,gender,semester,placement_test,course_id,level_id,class_id,question_id,version,text_len,text,tokens,tok_lem_POS
0,1,eq0,Arabic,Male,2006_fall,NaN,149,4,g,5,1,177,I met my friend Nife while I was studying in a...,"['I', 'met', 'my', 'friend', 'Nife', 'while', ...","[('I', 'I', 'PRP'), ('met', 'meet', 'VBD'), ('..."
1,2,am8,Thai,Female,2006_fall,NaN,149,4,g,5,1,137,"Ten years ago, I met a women on the train betw...","['Ten', 'years', 'ago', ',', 'I', 'met', 'a', ...","[('Ten', 'ten', 'CD'), ('years', 'year', 'NNS'..."
2,3,dk5,Turkish,Female,2006_fall,NaN,115,4,w,12,1,63,In my country we usually don't use tea bags. F...,"['In', 'my', 'country', 'we', 'usually', 'do',...","[('In', 'in', 'IN'), ('my', 'my', 'PRP$'), ('c..."


## 3. Inspect L1 coverage
See which L1s are available and how many texts each has.

In [30]:
l1_counts = df["L1"].value_counts()
print(f"Total unique L1s: {len(l1_counts)}\n")
print(l1_counts.to_string())

Total unique L1s: 30

L1
Arabic               16831
Korean                9208
Chinese               8503
Japanese              2782
Spanish               1909
Turkish               1537
Thai                  1376
Taiwanese              678
Portuguese             603
Other                  493
French                 477
Italian                393
Russian                193
Hebrew                 189
English                154
Farsi                  144
Mongol                 144
Vietnamese             104
German                  85
Indonesian              71
Romanian                69
Russian,Ukrainian       62
Azerbaijani             45
Suundi                  41
Swedish                 31
Montenegrin             27
Zulu                    25
Polish                  20
Hindi                    6
Swahili                  4


## 4. Configure target L1s
Edit this cell to match exactly what the L1 column uses (check the output above for exact spelling).

In [32]:
# Adjust these strings to match the exact L1 labels in the corpus
TARGET_L1S = {
    "spanish": "Spanish",
    "french":  "French",
    "italian": "Italian",
}

# Text column — PELIC uses 'text' but confirm from your column list above
TEXT_COL = "text"

# Minimum token count per text — filters out very short submissions
MIN_TOKENS = 50

## 5. Filter, clean, and report

In [34]:
def basic_clean(text):
    """Minimal cleaning: strip whitespace, drop empty."""
    if not isinstance(text, str):
        return None
    text = text.strip()
    return text if len(text) > 0 else None

subsets = {}

for label, l1_value in TARGET_L1S.items():
    subset = df[df["L1"] == l1_value][["L1", TEXT_COL]].copy()
    subset[TEXT_COL] = subset[TEXT_COL].apply(basic_clean)
    subset = subset.dropna(subset=[TEXT_COL])

    # Filter by minimum token count
    subset = subset[subset[TEXT_COL].str.split().str.len() >= MIN_TOKENS]

    subsets[label] = subset
    total_words = subset[TEXT_COL].str.split().str.len().sum()
    print(f"{label:10s} | texts: {len(subset):>5,} | total words: {total_words:>9,}")

spanish    | texts:   822 | total words:   175,467
french     | texts:   178 | total words:    42,789
italian    | texts:   108 | total words:    29,740


## 6. Inspect samples

In [36]:
for label, subset in subsets.items():
    print(f"\n--- {label.upper()} sample ---")
    print(subset[TEXT_COL].iloc[0][:400])
    print("...")


--- SPANISH sample ---
My mother is the most beautiful person that I know. 
First of all her facial expression is delicate and peaceful and she always is smiling.Her facial shape is oval and she has a big beautiful brown eyes and her mouth has full lipped.In adition, her voice is melodious and she talk slowly and she usually give good advice to other people like my father, me or my sister.I think that she always looks g
...

--- FRENCH sample ---
To make tea, nothing is easier, even if sometimes it could be dangerous. First, you need to have hot water. I let you choose the way that you make it. next, you need to buy a box of tea in wallmart or giant eagle. And finally you must use a beautiful cup to put the water and a small pack of tea in. You could also add sugar or/and milk. Don't forget to wait a few minutes before drinking, the tea co
...

--- ITALIAN sample ---
Learn another language, know about other culture, live in other country, look good a future resume, living with people 

## 7. Export per-L1 text files
Saves one `.txt` file per L1 (one text per line) and one combined `.csv` with an `l1` column — both formats useful downstream.

In [38]:
import os

os.makedirs("pelic_filtered", exist_ok=True)

all_frames = []

for label, subset in subsets.items():
    # One text per line .txt
    out_txt = f"pelic_filtered/{label}.txt"
    with open(out_txt, "w", encoding="utf-8") as f:
        for text in subset[TEXT_COL]:
            f.write(text.replace("\n", " ") + "\n")
    print(f"Saved {out_txt}")

    # Add l1 label for combined CSV
    tagged = subset[[TEXT_COL]].copy()
    tagged["l1"] = label
    all_frames.append(tagged)

# Combined CSV
combined = pd.concat(all_frames, ignore_index=True)
combined.to_csv("pelic_filtered/pelic_l1_combined.csv", index=False)
print(f"\nSaved combined CSV: {len(combined):,} texts across {combined['l1'].nunique()} L1s")

Saved pelic_filtered/spanish.txt
Saved pelic_filtered/french.txt
Saved pelic_filtered/italian.txt

Saved combined CSV: 1,108 texts across 3 L1s
